# Week 2, §3.3 — Mapping the equilibrium space

We classify each $(R, \kappa(\underline{\alpha}))$ pair by which equilibria
exist (separating, pool on $e=1$, pool on $e=0$, or none) using the analytical
conditions from §3.1--3.2.

Default parameters: $\kappa(\bar{\alpha}) = 1$, $K = 1.4$, $v = 1$,
$\underline{\alpha} = 1$, $\bar{\alpha} = 2$, $w(\bar{\alpha}) = 0.8$,
$w(\underline{\alpha}) = 0.3$, $p_H = 0.5$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
rng = np.random.default_rng(2026)


In [ ]:
def classify(R, kL, kH=1.0, K=1.4, v=1.0, aL=1.0, aH=2.0,
             wH=0.8, pH=0.5):
    Ealpha = pH * aH + (1 - pH) * aL
    sep = (kH <= R) and (R <= kL) and (v * aL < K) and (K <= v * aH) and (R >= wH)
    pool1 = (R >= kL) and (v * Ealpha >= K)
    pool0 = (v * Ealpha < K)
    if sep and pool0:
        return 'sep+pool0'
    if sep and pool1:
        return 'sep+pool1'
    if sep:
        return 'separating'
    if pool1:
        return 'pool e=1'
    if pool0:
        return 'pool e=0'
    return 'neither'

labels = ['neither', 'separating', 'pool e=1', 'pool e=0',
          'sep+pool1', 'sep+pool0']
label_to_int = {l: i for i, l in enumerate(labels)}
cmap = ListedColormap(['lightgray', 'C2', 'C0', 'C3', 'C5', 'gold'])


## (a) Partition over $(R, \kappa(\underline{\alpha}))$ at $K = 1.4$


In [ ]:
Rs = np.linspace(0, 5, 200)
kLs = np.linspace(1, 5, 200)
RR, KK = np.meshgrid(Rs, kLs)
grid = np.vectorize(lambda R, kL: label_to_int[classify(R, kL)])(RR, KK)

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(grid, origin='lower', extent=(0, 5, 1, 5),
               cmap=cmap, aspect='auto', vmin=-0.5, vmax=5.5)
cbar = plt.colorbar(im, ticks=range(6))
cbar.ax.set_yticklabels(labels)
ax.set(xlabel='$R$', ylabel=r'$\kappa(\underline{\alpha})$',
       title='Equilibrium partition at $K = 1.4$')
plt.tight_layout(); plt.show()


## (b) Boundary verification

The separating region is bounded by:

* $R \ge \kappa(\bar{\alpha}) = 1$ (high-type IC);
* $R \le \kappa(\underline{\alpha})$ (low-type no-mimic);
* $R \ge w(\bar{\alpha}) = 0.8$ (acceptance, slack here);
* $K \le v\bar{\alpha}$, $K > v\underline{\alpha}$ (party-side, slack at $K = 1.4$).


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(grid, origin='lower', extent=(0, 5, 1, 5),
               cmap=cmap, aspect='auto', vmin=-0.5, vmax=5.5, alpha=0.7)
ax.axvline(1.0, color='k', linestyle='--', alpha=0.7,
           label=r'$R = \kappa(\bar{\alpha})$')
ax.plot(Rs, Rs, 'k:', label=r'$R = \kappa(\underline{\alpha})$')
ax.set(xlabel='$R$', ylabel=r'$\kappa(\underline{\alpha})$',
       xlim=(0, 5), ylim=(1, 5),
       title='Theoretical boundaries overlaid')
ax.legend(loc='lower right'); plt.tight_layout(); plt.show()


## (c) Vary $K \in \{0.8, 1.4, 1.6, 2.5\}$


In [ ]:
Ks = [0.8, 1.4, 1.6, 2.5]
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, K in zip(axes, Ks):
    g = np.vectorize(lambda R, kL: label_to_int[classify(R, kL, K=K)])(RR, KK)
    im = ax.imshow(g, origin='lower', extent=(0, 5, 1, 5),
                   cmap=cmap, aspect='auto', vmin=-0.5, vmax=5.5)
    ax.set(xlabel='$R$', ylabel=r'$\kappa(\underline{\alpha})$',
           title=f'$K = {K}$')
fig.suptitle('Partitions across $K$ values', y=1.02)
plt.tight_layout(); plt.show()
print('Thresholds:')
print('  K = v*alpha_low      = 1.0   (below: party-side fails)')
print('  K = v*E[alpha]       = 1.5   (above: pool e=0 becomes available)')
print('  K = v*alpha_high     = 2.0   (above: even high posterior insufficient)')


## (d) Vary $w(\bar{\alpha}) \in [0, 3]$ holding $R = 1.5$


In [ ]:
wH_grid = np.linspace(0, 3, 121)
kL_grid = np.linspace(1, 5, 121)
WH, KL = np.meshgrid(wH_grid, kL_grid)
g_w = np.vectorize(
    lambda w, kL: label_to_int[classify(R=1.5, kL=kL, wH=w)]
)(WH, KL)
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(g_w, origin='lower', extent=(0, 3, 1, 5),
               cmap=cmap, aspect='auto', vmin=-0.5, vmax=5.5)
cbar = plt.colorbar(im, ticks=range(6))
cbar.ax.set_yticklabels(labels)
ax.axvline(1.5, color='k', linestyle='--',
           label=r'$w(\bar{\alpha}) = R = 1.5$ — separation collapses')
ax.set(xlabel=r'$w(\bar{\alpha})$', ylabel=r'$\kappa(\underline{\alpha})$',
       title='Outside-option sweep at $R = 1.5$')
ax.legend(); plt.tight_layout(); plt.show()


## (e) Partial-pooling simulation

Pick parameters in the partial-pooling region (e.g., $K$ slightly above
$v\underline{\alpha}$), simulate the agent strategy, and estimate
$\Pr(\bar{\alpha} \mid \text{promoted})$.


In [ ]:
# parameters chosen so partial pooling is the candidate equilibrium
kH, kL, K = 1.0, 2.0, 1.2
v, aL, aH, pH = 1.0, 1.0, 2.0, 0.5
R = 1.5  # in the IC interval [kH, kL]

# party's indifference belief and low-type's mixing prob
mu_star = (K / v - aL) / (aH - aL)
rho_star = kL / R  # actually only sensible for R >= kL; here separating exists too
q_star = pH * (1 - mu_star) / ((1 - pH) * mu_star) if mu_star > 0 else 0.0
print(f'mu* = {mu_star:.3f}, q* = {q_star:.3f}')

# simulate 1000 agents under partial-pooling strategy
N = 10_000
alphas = rng.choice([aH, aL], size=N, p=[pH, 1 - pH])
# strategy: high type plays e=1; low type plays e=1 with prob q*
u = rng.uniform(size=N)
exerted = (alphas == aH) | ((alphas == aL) & (u < q_star))
# party promotes after e=1 with probability rho* (mixed strategy)
promote_prob = rho_star
promoted = exerted & (rng.uniform(size=N) < promote_prob)
share_competent = (alphas[promoted] == aH).mean() if promoted.any() else float('nan')
print(f'Among {promoted.sum()} promoted agents, share competent = {share_competent:.3f}')
print(f'Theoretical mu* = {mu_star:.3f} (should match)')
